In [ ]:
import pandas as pd
import numpy as np
import re
from scipy.sparse import lil_matrix
from sklearn.model_selection import train_test_split

data = pd.read_csv(r'c:\Users\obasi\Downloads\archive (3)\IMDB Dataset.csv', encoding= 'latin1')
print(data.head(3))
data.shape
data.info()

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [2]:
data['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [3]:
print(data['review'][0])
print("/n" + "="*50 + "/n")
print(f"length: {len(data['review'][0])} characters")

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fac

In [4]:
def clean_review(text):
    text = text.lower()
    text = re.sub(r'<.*?>', ' ', text)  # Remove HTML tags
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)  # Remove punctuation and special characters
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra whitespace
    text = re.sub(r'\d+', '', text)  # Remove digits
    tokens = text.split()
    return tokens

In [5]:
# Test on a sample review
sample = data['review'][0]
print("ORIGINAL:")
print(sample[:200])  # First 200 chars
print("\n" + "="*50 + "\n")
print("CLEANED:")
print(clean_review(sample)[:200])

ORIGINAL:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me abo


CLEANED:
['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', 'oz', 'episode', 'you', 'll', 'be', 'hooked', 'they', 'are', 'right', 'as', 'this', 'is', 'exactly', 'what', 'happened', 'with', 'me', 'the', 'first', 'thing', 'that', 'struck', 'me', 'about', 'oz', 'was', 'its', 'brutality', 'and', 'unflinching', 'scenes', 'of', 'violence', 'which', 'set', 'in', 'right', 'from', 'the', 'word', 'go', 'trust', 'me', 'this', 'is', 'not', 'a', 'show', 'for', 'the', 'faint', 'hearted', 'or', 'timid', 'this', 'show', 'pulls', 'no', 'punches', 'with', 'regards', 'to', 'drugs', 'sex', 'or', 'violence', 'its', 'is', 'hardcore', 'in', 'the', 'classic', 'use', 'of', 'the', 'word', 'it', 'is', 'called', 'oz', 'as', 'that', 'is', 'the', 'nickname',

In [6]:
# Create a small test sample
data_sample = data.head(100).copy()

# Apply your cleaning function
data_sample['cleaned_review'] = data_sample['review'].apply(clean_review)

# Let's inspect the results
print("Sample of cleaned reviews:")
print(data_sample['cleaned_review'].head(3))
print("\n" + "="*50 + "\n")

# Check if any reviews failed to process
print(f"Original reviews: {len(data_sample)}")
print(f"Cleaned reviews: {data_sample['cleaned_review'].notna().sum()}")

Sample of cleaned reviews:
0    [one, of, the, other, reviewers, has, mentione...
1    [a, wonderful, little, production, the, filmin...
2    [i, thought, this, was, a, wonderful, way, to,...
Name: cleaned_review, dtype: object


Original reviews: 100
Cleaned reviews: 100


In [7]:
print('Processing the entire dataset...')
data['cleaned_review'] = data['review'].apply(clean_review)
print("Processing complete.")

print(f"Original reviews: {len(data)}")
print(f"Cleaned reviews: {data['cleaned_review'].notna().sum()}")   

Processing the entire dataset...
Processing complete.
Original reviews: 50000
Cleaned reviews: 50000


In [8]:
all_tokens =[]
for tokens in data['cleaned_review']:
    all_tokens.extend(tokens)
print(f"Total tokens in dataset: {len(all_tokens)}")

vocabulary = sorted( set(all_tokens))
print(f"Unique tokens (vocabulary size): {len(vocabulary)}")
print(f"/n First 20 words in vocabulary: {list(vocabulary)[:20]} ")

Total tokens in dataset: 11709196
Unique tokens (vocabulary size): 99420
/n First 20 words in vocabulary: ['a', 'aa', 'aaa', 'aaaaaaaaaaaahhhhhhhhhhhhhh', 'aaaaaaaargh', 'aaaaaaah', 'aaaaaaahhhhhhggg', 'aaaaagh', 'aaaaah', 'aaaaahhhh', 'aaaaargh', 'aaaaarrrrrrgggggghhhhhh', 'aaaaatch', 'aaaaaw', 'aaaahhhhhh', 'aaaahhhhhhh', 'aaaand', 'aaaarrgh', 'aaaawwwwww', 'aaaggghhhhhhh'] 


In [9]:

# Step 1: Create word-to-index mapping
word_to_idx = {word: idx for idx, word in enumerate(vocabulary)}
print(f"Vocabulary mapped: {len(word_to_idx)} words")

# Step 2: Initialize sparse matrix
X = lil_matrix((len(data), len(vocabulary)), dtype=np.int32)
print(f"Matrix shape: {X.shape}")

# Step 3: Fill the matrix with word counts
print("\nProcessing reviews...")
for review_idx, tokens in enumerate(data['cleaned_review']):
    # Count word occurrences in this review
    for token in tokens:
        if token in word_to_idx:  # Safety check
            word_idx = word_to_idx[token]
            X[review_idx, word_idx] += 1
    
    # Progress indicator (every 10000 reviews)
    if (review_idx + 1) % 10000 == 0:
        print(f"Processed {review_idx + 1} reviews...")

print("\nDone!")
print(f"Matrix shape: {X.shape}")
print(f"Non-zero elements: {X.nnz}")

Vocabulary mapped: 99420 words
Matrix shape: (50000, 99420)

Processing reviews...
Processed 10000 reviews...
Processed 20000 reviews...
Processed 30000 reviews...
Processed 40000 reviews...
Processed 50000 reviews...

Done!
Matrix shape: (50000, 99420)
Non-zero elements: 6938083


In [10]:
from sklearn.model_selection import train_test_split

# Step 1: Prepare your labels (convert sentiment to numbers)
# positive = 1, negative = 0
y = data['sentiment'].map({'positive': 1, 'negative': 0})

print(f"Labels created: {len(y)}")
print(f"Positive reviews: {(y == 1).sum()}")
print(f"Negative reviews: {(y == 0).sum()}")

# Step 2: Split into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,      # 20% for testing
    random_state=42,    # For reproducibility
    stratify=y          # Keep same ratio of positive/negative in both sets
)

print(f"\nTraining set: {X_train.shape[0]} reviews")
print(f"Test set: {X_test.shape[0]} reviews")
print(f"\nPositive in train: {(y_train == 1).sum()}")
print(f"Negative in train: {(y_train == 0).sum()}")

Labels created: 50000
Positive reviews: 25000
Negative reviews: 25000

Training set: 40000 reviews
Test set: 10000 reviews

Positive in train: 20000
Negative in train: 20000


In [11]:
print(f"Positive in test: {(y_test == 1).sum()}")
print(f"Negative in test: {(y_test == 0).sum()}")

Positive in test: 5000
Negative in test: 5000


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Step 1: Create and train the model
print("Training the model...")
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
print("Training complete!")

# Step 2: Make predictions on test set
print("\nMaking predictions on test set...")
y_pred = model.predict(X_test)

# Step 3: Evaluate performance
accuracy = accuracy_score(y_test, y_pred)
print(f"\nTest Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

# Step 4: Detailed performance report
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

Training the model...
Training complete!

Making predictions on test set...

Test Accuracy: 0.8935 (89.35%)

Detailed Classification Report:
              precision    recall  f1-score   support

    Negative       0.90      0.89      0.89      5000
    Positive       0.89      0.90      0.89      5000

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



In [13]:
def predict_sentiment(review_text):
    # Clean the review
    cleaned = clean_review(review_text)
    
    # Convert to BoW vector
    review_vector = lil_matrix((1, len(vocabulary)), dtype=np.int32)
    for token in cleaned:
        if token in word_to_idx:
            review_vector[0, word_to_idx[token]] += 1
    
    # Predict
    prediction = model.predict(review_vector)[0]
    probability = model.predict_proba(review_vector)[0]
    
    sentiment = "Positive" if prediction == 1 else "Negative"
    confidence = probability[prediction] * 100
    
    print(f"Review: '{review_text}'")
    print(f"Prediction: {sentiment} ({confidence:.1f}% confident)")
    print()

# Test it!
predict_sentiment("This movie was absolutely amazing! I loved every minute of it.")
predict_sentiment("Waste of time. Boring and predictable.")
predict_sentiment("The acting was great but the plot was terrible.")

Review: 'This movie was absolutely amazing! I loved every minute of it.'
Prediction: Positive (87.4% confident)

Review: 'Waste of time. Boring and predictable.'
Prediction: Negative (98.6% confident)

Review: 'The acting was great but the plot was terrible.'
Prediction: Negative (68.2% confident)

